In [0]:
import re
import pyspark.sql.functions as f
import pyspark.sql.types as t
from functools import reduce

In [0]:
def clean_column_name(col_name):
    """
    Remove or replace invalid characters from a column name.
    Databricks Delta doesn't allow: space , ; { } ( ) \n \t =
    
    Args:
        col_name: Original column name
        
    Returns:
        Cleaned column name
    """
    # Replace invalid characters with underscores
    invalid_chars = r'[ ,;{}()\n\t=]'
    cleaned = re.sub(invalid_chars, '_', col_name)
    
    # Remove consecutive underscores
    cleaned = re.sub(r'_+', '_', cleaned)
    
    # Remove leading/trailing underscores
    cleaned = cleaned.strip('_')
    
    # If empty after cleaning, use 'col_N' as fallback
    if not cleaned:
        cleaned = 'col_unnamed'
    
    return cleaned

def clean_dataframe_schema_dict(df):
    """
    Alternative approach using withColumnsRenamed (PySpark 3.4+).
    More efficient for many columns.
    
    Args:
        df: PySpark DataFrame with potentially invalid column names
        
    Returns:
        DataFrame with cleaned column names
    """
    rename_map = {
        old_col: clean_column_name(old_col) 
        for old_col in df.columns
    }
    
    # Filter to only columns that actually need renaming
    rename_map = {k: v for k, v in rename_map.items() if k != v}
    
    if rename_map:
        return df.withColumnsRenamed(rename_map)
    return df

def clean_df(df):
    clean_schema_df = clean_dataframe_schema_dict(df)
    string_cols = [col.name for col in clean_schema_df.schema if col.dataType.typeName() == 'string']
    return reduce(lambda x,y: x.withColumn(y, f.trim(y)), string_cols, clean_schema_df)